In [ ]:
import pyspark
import dxpy
import hail as hl
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import json
import seaborn as sns 
from scipy.stats import boxcox

In [ ]:
sys.path.append('../')
from functions.phenotype_calculation_utils import calculate_therapy_doses

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Configuration and Hail tables loading

In [ ]:
input_database = 'prescriptions_db'
input_prescriptions_tb = 'cleaned_prescriptions_splited_to_therapies_v6.2.0.ht'

output_database = 'prescriptions_db'
output_tb = 'pdc_v6_2_0.ht'

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
ht = hl.read_table(f'dnax://{input_db_id}/{input_prescriptions_tb}')
ht = ht.filter(~hl.is_missing(ht.date_struct.year))

In [ ]:
ht_summary = calculate_therapy_doses(ht)

In [ ]:
ht_summary = ht_summary.annotate(
    total_days_covered = hl.sum(
        ht_summary.sorted_records.map(
            lambda r: r.sum_quantity
        )
    )
)
ht_summary = ht_summary.persist()

In [ ]:
ht_summary = ht_summary.annotate(
    pdc = hl.if_else(
        ht_summary.duration > 0,
        hl.min(1.0, ht_summary.total_days_covered / ht_summary.duration) * 100.0,
        hl.missing(hl.tfloat64)
    )
)
ht_summary = ht_summary.persist()

In [ ]:
pdc_values = ht_summary.aggregate(hl.agg.collect(ht_summary.pdc))
pdc_array = np.array(pdc_values)

In [ ]:
pdc_clean = pdc_array[~np.isnan(pdc_array)]
plt.figure(figsize=(10, 6))
bins = np.arange(0, 105, 5) 

plt.hist(
    pdc_clean, 
    bins=bins, 
    edgecolor='black', 
    color='#606c38',
    alpha=0.8
)

plt.title('Proportion of Days Covered (PDC)')
plt.xlabel('PDC (%)')
plt.ylabel('Frequency')
plt.xticks(bins) 
plt.xlim(0, 100) 
plt.grid(axis='y', alpha=0.5, linestyle='--')

plt.tight_layout()
plt.show() 

In [ ]:
ht_summary = ht_summary.annotate(
    pdc_weighted_value = ht_summary.pdc * ht_summary.duration
)
ht_patient_summary = ht_summary.group_by(
    ht_summary.eid 
).aggregate(
    mean_pdc = hl.agg.mean(ht_summary.pdc),
    weighted_mean_pdc = hl.agg.sum(ht_summary.pdc_weighted_value) / hl.agg.sum(ht_summary.duration),
)
ht_patient_summary=ht_patient_summary.persist()

In [ ]:
pdc_values = ht_patient_summary.aggregate(hl.agg.collect(ht_patient_summary.weighted_mean_pdc))
pdc_array = np.array(pdc_values)

In [ ]:
pdc_clean = pdc_array[~np.isnan(pdc_array)]
plt.figure(figsize=(10, 6))
bins = np.arange(0, 105, 5) 

plt.hist(
    pdc_clean, 
    bins=bins, 
    edgecolor='black', 
    color='#606c38',
    alpha=0.8
)

plt.title('Proportion of Days Covered (PDC)')
plt.xlabel('PDC (%)')
plt.ylabel('Frequency')
plt.xticks(bins) 
plt.xlim(0, 100) 
plt.grid(axis='y', alpha=0.5, linestyle='--')

plt.tight_layout()
plt.show() 

In [ ]:
ht_patient_summary = ht_patient_summary.key_by().select('mean_pdc','weighted_mean_pdc','eid')

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {output_database} LOCATION 'dnax://'")
output_db_id = dxpy.find_one_data_object(name=output_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
output_url = f'dnax://{output_db_id}/{output_tb}'

%time ht_patient_summary.key_by('eid').write(output_url, overwrite=True)